<a href="https://colab.research.google.com/github/Jayantsinghkhanna/UCS547-ACCELERATED-DATA-SCIENCE-LAB/blob/main/Assignmen_2_AcceleratedDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Q1
# !  -> runs Linux shell commands
!pwd
!ls

# %  -> single line magic
%time x = sum(range(1000000))

# %% -> entire cell magic


/content
sample_data
CPU times: user 21.9 ms, sys: 0 ns, total: 21.9 ms
Wall time: 22.5 ms


In [4]:
print("1) Basic GPU Status (default nvidia-smi output):")
!nvidia-smi

print("\n2) List all available GPUs (GPU name + UUID):")
!nvidia-smi -L

print("\n3) Query specific GPU information in CSV format (name, driver, temp, utilization, memory usage):")
!nvidia-smi --query-gpu=name,driver_version,temperature.gpu,utilization.gpu,memory.used,memory.total --format=csv

print("\n4) Full detailed GPU report (all hardware + software details):")
!nvidia-smi -q

print("\n5) Detailed GPU Memory information only:")
!nvidia-smi -q -d MEMORY

print("\n6) Show running processes using GPU (process monitoring):")
!nvidia-smi pmon -c 1


1) Basic GPU Status (default nvidia-smi output):
Wed Feb 18 04:52:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |

In [6]:
print("\n7) Live GPU monitoring every 1 second (Press CTRL+C to stop):")
!nvidia-smi -l 1


7) Live GPU monitoring every 1 second (Press CTRL+C to stop):
Wed Feb 18 04:53:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |         

In [23]:
%%writefile debug_sync.cu
// Zero output error
#include <stdio.h>

__global__ void addKernel(int *c, int a, int b) {
    *c = a + b;
}

int main() {
    int h_c = 0;
    int *d_c;
    cudaMalloc(&d_c, sizeof(int));

    addKernel<<<1, 1>>>(d_c, 10, 20);

    cudaDeviceSynchronize(); // CRITICAL: Without this, h_c will be 0 because memcpy might trigger before the kernel is actually done on the hardware.

    cudaMemcpy(&h_c, d_c, sizeof(int), cudaMemcpyDeviceToHost);
    printf("Result: %d\n", h_c); // Output: 30

    cudaFree(d_c);
    return 0;
}

Overwriting debug_sync.cu


In [24]:
!nvcc debug_sync.cu -o debug_sync

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [25]:
!./debug_sync

Result: 30


In [68]:
%%writefile correct_index.cu
#include<stdio.h>

__global__ void kernel(){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Thread ID: %d\n", id);
}

int main(){
    kernel<<<2,4>>>();
    cudaDeviceSynchronize();
    return 0;
}


Overwriting correct_index.cu


In [69]:
!nvcc -arch=sm_75 correct_index.cu -o correct_index

In [72]:
!./correct_index

Thread ID: 0
Thread ID: 1
Thread ID: 2
Thread ID: 3
Thread ID: 4
Thread ID: 5
Thread ID: 6
Thread ID: 7


In [71]:
!nvcc -arch=sm_75 correct_index.cu -o correct_index

In [7]:
%%writefile q4_thread_indexing.cu
#include <stdio.h>
#include <cuda_runtime.h>

// -------------------- Device Code (GPU Kernel) --------------------
__global__ void helloKernel()
{
    int global_thread_id = blockIdx.x * blockDim.x + threadIdx.x;

    printf("Hello from GPU thread %d\n", global_thread_id);
}

// -------------------- Host Code (CPU Code) --------------------
int main()
{
    printf("Hello from CPU (Host Code)\n");

    // Kernel launch: 1 block and 8 threads
    helloKernel<<<1, 8>>>();

    // Synchronize CPU with GPU
    cudaDeviceSynchronize();

    return 0;
}


Writing q4_thread_indexing.cu


In [8]:
!nvcc q4_thread_indexing.cu -o q4_thread_indexing


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [9]:
!./q4_thread_indexing


Hello from CPU (Host Code)
Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from GPU thread 4
Hello from GPU thread 5
Hello from GPU thread 6
Hello from GPU thread 7


In [10]:
%%writefile q5_host_device_memory.cu
#include <stdio.h>
#include <cuda_runtime.h>

// -------------------- Device Code (GPU Kernel) --------------------
__global__ void printDeviceArray(int *d_arr)
{
    int id = blockIdx.x * blockDim.x + threadIdx.x;

    if (id < 5)
    {
        printf("GPU Thread %d reads d_arr[%d] = %d\n", id, id, d_arr[id]);
    }
}

// -------------------- Host Code (CPU Code) --------------------
int main()
{
    int h_arr[5] = {10, 20, 30, 40, 50};
    int *d_arr;

    printf("Host Array Values (CPU Memory):\n");
    for (int i = 0; i < 5; i++)
    {
        printf("h_arr[%d] = %d\n", i, h_arr[i]);
    }

    // Allocate memory on GPU
    cudaMalloc((void**)&d_arr, 5 * sizeof(int));

    // Copy Host -> Device
    cudaMemcpy(d_arr, h_arr, 5 * sizeof(int), cudaMemcpyHostToDevice);

    printf("\nLaunching kernel to print values from GPU memory...\n");

    // Kernel launch: 1 block, 5 threads
    printDeviceArray<<<1, 5>>>(d_arr);

    cudaDeviceSynchronize();

    // Copy Device -> Host
    cudaMemcpy(h_arr, d_arr, 5 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nCopied back to Host (CPU) after cudaMemcpy:\n");
    for (int i = 0; i < 5; i++)
    {
        printf("h_arr[%d] = %d\n", i, h_arr[i]);
    }

    // Free device memory
    cudaFree(d_arr);

    return 0;
}


Writing q5_host_device_memory.cu


In [11]:
!nvcc q5_host_device_memory.cu -o q5_host_device_memory


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [12]:
!./q5_host_device_memory


Host Array Values (CPU Memory):
h_arr[0] = 10
h_arr[1] = 20
h_arr[2] = 30
h_arr[3] = 40
h_arr[4] = 50

Launching kernel to print values from GPU memory...
GPU Thread 0 reads d_arr[0] = 10
GPU Thread 1 reads d_arr[1] = 20
GPU Thread 2 reads d_arr[2] = 30
GPU Thread 3 reads d_arr[3] = 40
GPU Thread 4 reads d_arr[4] = 50

Copied back to Host (CPU) after cudaMemcpy:
h_arr[0] = 10
h_arr[1] = 20
h_arr[2] = 30
h_arr[3] = 40
h_arr[4] = 50


In [13]:
import time
import numpy as np

N = 10_000_00   # 1 million (increase if needed)

print("--------- CPU Time Comparison ---------\n")

# ---------------- LIST ----------------
lst = list(range(N))
start = time.time()
list_sum = sum(lst)
end = time.time()
print("List sum time:", end - start, "seconds")

# ---------------- TUPLE ----------------
tup = tuple(range(N))
start = time.time()
tuple_sum = sum(tup)
end = time.time()
print("Tuple sum time:", end - start, "seconds")

# ---------------- NUMPY ARRAY ----------------
arr = np.arange(N)
start = time.time()
numpy_sum = np.sum(arr)
end = time.time()
print("NumPy sum time:", end - start, "seconds")

print("\n--------- Results ---------")
print("List Sum  :", list_sum)
print("Tuple Sum :", tuple_sum)
print("NumPy Sum :", numpy_sum)


--------- CPU Time Comparison ---------

List sum time: 0.007918119430541992 seconds
Tuple sum time: 0.00775909423828125 seconds
NumPy sum time: 0.0006871223449707031 seconds

--------- Results ---------
List Sum  : 499999500000
Tuple Sum : 499999500000
NumPy Sum : 499999500000
